In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

# Đồng bộ hệ thống đường dẫn tương đối
sys.path = [p for p in sys.path if p is not None]
src_path = os.getcwd() 
if src_path not in sys.path:
    sys.path.append(src_path)

# Import tracker và hàm load dữ liệu chuẩn của nhóm
from src.internal_tracker import ExperimentTracker
from src.data_pipeline import load_raw_data

# Khởi tạo tracker v6
tracker = ExperimentTracker(project_root=src_path, version="v6")
print("✅ BƯỚC 1 HOÀN THÀNH: Đã kích hoạt hệ thống Tracker cho Version 6!")

✅ BƯỚC 1 HOÀN THÀNH: Đã kích hoạt hệ thống Tracker cho Version 6!


In [2]:
print("⏳ Đang thu gom các mảng dự đoán (.npy) từ Version 5...")

try:
    # Đọc các file mảng xác suất nền tảng
    oof_lgb = np.load('oof_lgb_v5.npy')
    oof_xgb = np.load('oof_xgb_v5.npy')
    oof_cat = np.load('oof_cat_v5.npy')
    
    # Xếp chồng các ma trận xác suất lại theo hàng ngang
    X_train_meta = np.hstack([oof_lgb, oof_xgb, oof_cat])
    print(f"➔ Kích thước đặc trưng Meta-Train: {X_train_meta.shape}")
    
except FileNotFoundError:
    print("❌ Lỗi: Không tìm thấy các file mảng .npy của V5! Hãy chắc chắn các file .npy nằm cùng thư mục với notebook này.")
    raise

# Đọc nhãn thực tế từ thư mục data để làm đáp án huấn luyện
data_dir = os.path.join(src_path, 'data')
train_df, test_df, _ = load_raw_data(data_dir)
y_train = train_df['Class']

print("✅ BƯỚC 2 HOÀN THÀNH: Đã gộp nguyên liệu Meta-features thành công!")

⏳ Đang thu gom các mảng dự đoán (.npy) từ Version 5...
➔ Kích thước đặc trưng Meta-Train: (14396, 33)
✅ BƯỚC 2 HOÀN THÀNH: Đã gộp nguyên liệu Meta-features thành công!


In [3]:
print("🧠 Đang huấn luyện mô hình trọng tài Stacking...")

# Khởi tạo mô hình tuyến tính phân xử, bật balanced để giữ vững phong độ xử lý imbalance
meta_model = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=42)
meta_model.fit(X_train_meta, y_train)

# Đánh giá điểm số Out-of-Fold của hệ thống Stacking
meta_oof_preds = meta_model.predict(X_train_meta)
v6_macro_f1 = f1_score(y_train, meta_oof_preds, average='macro')

print("\n================ KẾT QUẢ ĐỈNH CAO VERSION 6 ================")
print(f"🎯 Điểm số F1 Macro sau Stacking Ensemble: {v6_macro_f1:.5f}")
print("============================================================")
print(classification_report(y_train, meta_oof_preds))

# Tự động đẩy điểm số và file báo cáo tổng kết cuối cùng vào hệ thống của nhóm
tracker.log_metrics({"v6_final_meta_macro_f1": v6_macro_f1})
tracker.log_text_report("final_report.txt", classification_report(y_train, meta_oof_preds))

print("✅ BƯỚC 3 HOÀN THÀNH: Đã ghi nhận báo cáo tổng kết vào file 'final_report.txt'!")

🧠 Đang huấn luyện mô hình trọng tài Stacking...

================ KẾT QUẢ ĐỈNH CAO VERSION 6 ================
🎯 Điểm số F1 Macro sau Stacking Ensemble: 0.58491
              precision    recall  f1-score   support

           0       0.67      0.73      0.70       500
           1       0.22      0.36      0.28      1098
           2       0.40      0.64      0.49      1018
           3       0.75      0.79      0.77       322
           4       0.58      0.80      0.67       310
           5       0.66      0.74      0.70      1157
           6       0.42      0.36      0.39      2069
           7       0.94      0.94      0.94       461
           8       0.52      0.75      0.61      1483
           9       0.53      0.51      0.52      2019
          10       0.55      0.29      0.38      3959

    accuracy                           0.50     14396
   macro avg       0.57      0.63      0.58     14396
weighted avg       0.52      0.50      0.49     14396

✅ BƯỚC 3 HOÀN THÀNH: Đã ghi

In [7]:
print("⏳ Đang khởi tạo và cấu hình lại Pipeline cho tập Test...")

import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.utils.class_weight import compute_sample_weight

# --- TÁCH BẠCH RÕ RÀNG IMPORT TỪ HAI FILE KHÁC NHAU ---
from src.data_pipeline import handle_missing_values
from src.feature_engineering import apply_log_transform, remove_high_correlation, select_top_features

# 1. Chuẩn bị dữ liệu thô ban đầu
X_tr_raw = train_df.drop(columns=['Id', 'Class', 'Artist Name', 'Track Name'], errors='ignore')
X_te_raw = test_df.drop(columns=['Id', 'Artist Name', 'Track Name'], errors='ignore')

# 2. Bước xử lý khuyết thiếu (từ data_pipeline)
X_tr_f = handle_missing_values(X_tr_raw)
X_te_f = handle_missing_values(X_te_raw)

# 3. Bước biến đổi Log-transform (từ feature_engineering)
X_tr_l = apply_log_transform(X_tr_f, columns=['speechiness', 'acousticness', 'instrumentalness'])
X_te_l = apply_log_transform(X_te_f, columns=['speechiness', 'acousticness', 'instrumentalness'])

# 4. Bước loại bỏ đặc trưng tương quan cao (từ feature_engineering)
X_tr_nocorr, dropped_features = remove_high_correlation(X_tr_l, threshold=0.90)
X_te_nocorr = X_te_l.drop(columns=dropped_features, errors='ignore')

# 5. Bước trích chọn đặc trưng mạnh nhất SelectKBest (từ feature_engineering)
X_tr_selected, selector = select_top_features(X_tr_nocorr, y_train, k=10)
X_te_selected = selector.transform(X_te_nocorr)

print("⏳ Đang thu thập ma trận xác suất tập Test từ các mô hình nền tảng...")

# 6. Huấn luyện thực tế trên toàn bộ tập dữ liệu scale
m_lgb = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300, verbose=-1).fit(X_tr_selected, y_train)
m_xgb = xgb.XGBClassifier(random_state=42, n_estimators=300, eval_metric='mlogloss').fit(X_tr_selected, y_train, sample_weight=compute_sample_weight(class_weight='balanced', y=y_train))
m_cat = CatBoostClassifier(auto_class_weights='Balanced', random_state=42, iterations=300, verbose=0).fit(X_tr_selected, y_train)

# 7. Xếp chồng tạo Meta-features cho tập Test
X_test_meta = np.hstack([m_lgb.predict_proba(X_te_selected), m_xgb.predict_proba(X_te_selected), m_cat.predict_proba(X_te_selected)])

# 8. Dự đoán nhãn cuối cùng từ mô hình trọng tài Stacking
final_predictions = meta_model.predict(X_test_meta)

# 9. Đóng gói sản phẩm nộp bài theo đúng cấu trúc mẫu
submission_v6 = pd.DataFrame({
    'Id': test_df['Id'],
    'Class': final_predictions
})

# Kiểm tra an toàn số lượng dòng tập dữ liệu
assert len(submission_v6) == 3600, "Cảnh báo: Số dòng tập test bị lệch!"
submission_file_v6 = 'final_submission_version6.csv'
submission_v6.to_csv(submission_file_v6, index=False)

print(f"\n🎉🎉🎉 ĐỒ ÁN HOÀN THÀNH MỸ MÃN KHÔNG CÒN LỖI!")
print(f"👉 File nộp bài cuối cùng của nhóm nằm tại: '{os.path.abspath(submission_file_v6)}'")
print("🏁 Xin chúc mừng! Nhiệm vụ đồ án của Người 3 đã kết thúc tốt đẹp!")

⏳ Đang khởi tạo và cấu hình lại Pipeline cho tập Test...
⏳ Đang thu thập ma trận xác suất tập Test từ các mô hình nền tảng...

🎉🎉🎉 ĐỒ ÁN HOÀN THÀNH MỸ MÃN KHÔNG CÒN LỖI!
👉 File nộp bài cuối cùng của nhóm nằm tại: 'c:\Users\HI\nhatkykag\final_submission_version6.csv'
🏁 Xin chúc mừng! Nhiệm vụ đồ án của Người 3 đã kết thúc tốt đẹp!


c:\Users\HI\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
